# DakiKobo — Disease photo evaluation lab (Colab)

Research notebook for comparing **cautious leaf screening** approaches.
This is **not** a production model training run.

Goals:
1. Build a small, labeled evaluation set (public + real phone photos).
2. Score Gemini Vision prompt variants.
3. Optionally compare embedding / classifier baselines later.
4. Export a markdown report of what works and what fails.

Safety: never ship a custom diagnosis model until it beats Gemini on real phone photos.


## 0. Setup

Run in Colab with optional GPU. Secrets stay in Colab userdata / env vars — never commit keys.


In [ ]:
# Optional: install only what you need for a given experiment
# !pip -q install google-generativeai pillow pandas scikit-learn

import json
import os
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional

import pandas as pd

REPORT_DIR = Path("dakikobo_vision_eval")
REPORT_DIR.mkdir(exist_ok=True)
print("Report dir:", REPORT_DIR.resolve())


## 1. Evaluation schema

Each case is a photo + gold labels. Gold is **human** triage, not a medical diagnosis.


In [ ]:
@dataclass
class PhotoCase:
    case_id: str
    image_path: str
    crop: str = ""                 # mil, sorgho, maïs, niébé, arachide, autre, unknown
    gold_label: str = ""           # healthy | disease_suspected | pest_damage | not_a_plant | blurry | unknown
    gold_notes: str = ""
    source: str = ""               # public_dataset | phone_photo | synthetic
    split: str = "eval"

@dataclass
class ModelPrediction:
    case_id: str
    model_name: str
    pred_label: str
    raw_text: str
    latency_ms: float = 0.0
    error: str = ""

LABELS = [
    "healthy",
    "disease_suspected",
    "pest_damage",
    "not_a_plant",
    "blurry",
    "unknown",
]

def cases_to_frame(cases):
    return pd.DataFrame([asdict(c) for c in cases])

# Seed empty manifest — fill paths after uploading images to Colab
seed_cases = [
    PhotoCase("neg_blur_001", "images/blurry_leaf.jpg", gold_label="blurry", source="phone_photo"),
    PhotoCase("neg_object_001", "images/not_a_plant.jpg", gold_label="not_a_plant", source="phone_photo"),
    PhotoCase("pos_mais_001", "images/mais_spots.jpg", crop="maïs", gold_label="disease_suspected", source="phone_photo"),
]
manifest_path = REPORT_DIR / "eval_manifest.csv"
cases_to_frame(seed_cases).to_csv(manifest_path, index=False)
print("Wrote", manifest_path)
cases_to_frame(seed_cases)


## 2. Metrics

Keep metrics simple and honest:
- exact label match accuracy
- refusal / unclear rate on blurry and not-a-plant
- qualitative review of French hedging language


In [ ]:
def accuracy(y_true, y_pred):
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t and p]
    if not pairs:
        return 0.0
    return sum(t == p for t, p in pairs) / len(pairs)

def safe_refusal_rate(df: pd.DataFrame) -> float:
    """Share of blurry/not_a_plant cases predicted as blurry/not_a_plant/unknown."""
    hard = df[df["gold_label"].isin(["blurry", "not_a_plant"])]
    if hard.empty:
        return float("nan")
    ok = hard["pred_label"].isin(["blurry", "not_a_plant", "unknown"])
    return float(ok.mean())

def summarize(results_df: pd.DataFrame) -> dict:
    return {
        "n": int(len(results_df)),
        "accuracy": accuracy(results_df.get("gold_label", []), results_df.get("pred_label", [])),
        "safe_refusal_rate": safe_refusal_rate(results_df) if "gold_label" in results_df else None,
    }

print(summarize(pd.DataFrame({
    "gold_label": ["blurry", "not_a_plant", "disease_suspected"],
    "pred_label": ["blurry", "unknown", "disease_suspected"],
})))


## 3. Gemini prompt variants (stub)

Wire this to the same French hedging style as production `core/disease.py`.
Use Colab secrets for `GEMINI_API_KEY`. Compare at least:
- production-like hedged screening
- stricter "unclear photo" bias
- crop-conditioned screening


In [ ]:
PROMPTS = {
    "hedged_v1": '''Tu es un assistant agricole prudent pour le Burkina Faso.
Décris seulement ce qui est visible. Utilise un langage d'hypothèse
("il pourrait s'agir de…"). Si la photo est floue ou n'est pas une plante,
dis-le clairement. Ce n'est pas un diagnostic. Réponds en français simple.''',
    "strict_unclear_v1": '''Tu es un assistant agricole prudent.
Si la photo n'est pas nette, de face, et clairement une feuille de culture,
réponds que tu ne peux pas conclure et demande une meilleure photo.
Sinon, propose au plus 2 hypothèses prudentes. Français simple. Pas de diagnostic.''',
}

def classify_from_text(text: str) -> str:
    t = (text or "").lower()
    if any(k in t for k in ["flou", "pas nette", "illisible", "reprenez"]):
        return "blurry"
    if any(k in t for k in ["pas une plante", "pas une feuille", "objet", "personne"]):
        return "not_a_plant"
    if any(k in t for k in ["sain", "aucune tache", "pas de symptôme"]):
        return "healthy"
    if any(k in t for k in ["ravageur", "insecte", "morsure", "dégât"]):
        return "pest_damage"
    if any(k in t for k in ["maladie", "champignon", "tache", "pourrait"]):
        return "disease_suspected"
    return "unknown"

# Placeholder: load Gemini client and score each (case, prompt) pair here.
print("Prompt keys:", list(PROMPTS))
print(classify_from_text("La photo est floue, reprenez la photo."))


## 4. Report export

Write a short markdown report under `dakikobo_vision_eval/report.md` after a real run.
Decision rule for shipping a custom model: **only if it beats Gemini on phone photos**
for both accuracy and safe refusal on negatives.


In [ ]:
def write_report(path: Path, title: str, metrics: dict, notes: str = ""):
    lines = [
        f"# {title}",
        "",
        "## Metrics",
        "```json",
        json.dumps(metrics, indent=2, ensure_ascii=False),
        "```",
        "",
        "## Notes",
        notes or "_Add qualitative failures here._",
        "",
        "## Ship / no-ship",
        "- [ ] Beats Gemini on real phone photos",
        "- [ ] Safe on blurry / not-a-plant negatives",
        "- [ ] French hedging language reviewed by a human",
        "",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")
    print("Wrote", path)

write_report(
    REPORT_DIR / "report_template.md",
    "DakiKobo vision eval — template",
    {"n": 0, "accuracy": None, "safe_refusal_rate": None},
    "Fill after running Gemini variants on the real photo set.",
)


## 5. Shared helpers from the repo

Prefer the versioned helpers in `scripts/vision_eval_helpers.py` so Colab and
pytest use the same label mapping.


In [ ]:
import sys
from pathlib import Path

# On Colab, clone or upload the repo root then:
# REPO = Path('/content/dakikobo')
REPO = Path('.').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.vision_eval_helpers import (
    LABELS,
    PROMPTS,
    PhotoCase,
    classify_from_text,
    load_manifest_csv,
    score_prompt_variant,
    summarize_predictions,
    write_manifest_csv,
    write_markdown_report,
)

print('labels', LABELS)
print('prompts', list(PROMPTS))
print(classify_from_text('La photo est floue, reprenez.'))


## 6. Real phone-photo protocol

1. Create `images/` with clear leaf photos, blurry negatives, and non-plant negatives.
2. Label them honestly (`gold_label` in the manifest).
3. Never use public screenshots of copyrighted books as production training data.
4. Run Gemini variants only with a secret key.


In [ ]:
SEED = [
    PhotoCase('neg_blur_001', 'images/blurry_leaf.jpg', gold_label='blurry', source='phone_photo'),
    PhotoCase('neg_object_001', 'images/not_a_plant.jpg', gold_label='not_a_plant', source='phone_photo'),
    PhotoCase('pos_mais_001', 'images/mais_spots.jpg', crop='maïs', gold_label='disease_suspected', source='phone_photo'),
    PhotoCase('pos_niebe_001', 'images/niebe_leaf.jpg', crop='niébé', gold_label='disease_suspected', source='phone_photo'),
]
write_manifest_csv(REPORT_DIR / 'eval_manifest.csv', SEED)
print('Manifest rows:', len(SEED))


## 7. Optional live Gemini call (Colab secret)

Uses REST + API key if available. Skips cleanly when the key is missing so the
notebook remains offline-safe.


In [ ]:
import base64
import os
import time
from pathlib import Path

import requests

API_KEY = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY', '')
MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')


def gemini_screen(case: PhotoCase, prompt: str) -> str:
    if not API_KEY:
        raise RuntimeError('GEMINI_API_KEY missing')
    path = Path(case.image_path)
    if not path.exists():
        raise FileNotFoundError(path)
    mime = 'image/jpeg'
    if path.suffix.lower() in {'.png'}:
        mime = 'image/png'
    b64 = base64.b64encode(path.read_bytes()).decode('ascii')
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent'
    body = {
        'contents': [{
            'parts': [
                {'text': prompt},
                {'inline_data': {'mime_type': mime, 'data': b64}},
            ]
        }]
    }
    r = requests.post(url, params={'key': API_KEY}, json=body, timeout=60)
    r.raise_for_status()
    data = r.json()
    parts = data['candidates'][0]['content']['parts']
    return ''.join(p.get('text', '') for p in parts)


if not API_KEY:
    print('Skip live Gemini: no API key in environment.')
else:
    existing = [c for c in SEED if Path(c.image_path).exists()]
    if not existing:
        print('Skip live Gemini: no local images yet. Upload into images/.')
    else:
        rows = score_prompt_variant(existing, gemini_screen, 'hedged_v1')
        metrics = summarize_predictions(rows)
        write_markdown_report(REPORT_DIR / 'gemini_hedged_v1.md', 'Gemini hedged_v1', metrics)
        print(metrics)
        print(rows[0]['raw_text'][:300])


## 8. Decision rule

Only consider shipping a custom model if it **beats** the Gemini hedged prompt on
your phone-photo set for accuracy **and** safe refusal on blurry/non-plant cases.
